In [64]:
%pip install mysql-connector-python


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [74]:
import pandas as pd
import mysql.connector
from getpass import getpass
from pathlib import Path

In [75]:
csv_path = Path("../data/processed/retail_store_inventory_clean.csv")

df = pd.read_csv(csv_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (73100, 15)

Columns:
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality']


In [76]:
df["Date"] = pd.to_datetime(df["Date"])

print(df["Date"].min())
print(df["Date"].max())

2022-01-01 00:00:00
2024-01-01 00:00:00


In [77]:
print("Negative forecasts:", (df["Demand Forecast"] < 0).sum())

Negative forecasts: 0


In [78]:
mysql_password = getpass("Enter MySQL root password: ")

conn = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=mysql_password,
    database="retail_inventory_db"
)

cursor = conn.cursor()

print("MySQL connection successful!")

Enter MySQL root password:  ········


MySQL connection successful!


In [79]:
insert_query = """
INSERT INTO inventory_sales (
    date,
    store_id,
    product_id,
    category,
    region,
    inventory_level,
    units_sold,
    units_ordered,
    demand_forecast,
    price,
    discount,
    weather_condition,
    holiday_promotion,
    competitor_pricing,
    seasonality
)
VALUES (
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s
)
"""

In [80]:
records = list(
    df[
        [
            "Date",
            "Store ID",
            "Product ID",
            "Category",
            "Region",
            "Inventory Level",
            "Units Sold",
            "Units Ordered",
            "Demand Forecast",
            "Price",
            "Discount",
            "Weather Condition",
            "Holiday/Promotion",
            "Competitor Pricing",
            "Seasonality"
        ]
    ].itertuples(index=False, name=None)
)

print("Records prepared:", len(records))

Records prepared: 73100


In [81]:
batch_size = 5000

try:
    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]

        cursor.executemany(insert_query, batch)
        conn.commit()

        print(
            f"Inserted {min(start + batch_size, len(records)):,} "
            f"/ {len(records):,} rows"
        )

    print("\nData loading completed successfully!")

except Exception as e:
    conn.rollback()
    print("Error during data loading:")
    print(e)

Inserted 5,000 / 73,100 rows
Inserted 10,000 / 73,100 rows
Inserted 15,000 / 73,100 rows
Inserted 20,000 / 73,100 rows
Inserted 25,000 / 73,100 rows
Inserted 30,000 / 73,100 rows
Inserted 35,000 / 73,100 rows
Inserted 40,000 / 73,100 rows
Inserted 45,000 / 73,100 rows
Inserted 50,000 / 73,100 rows
Inserted 55,000 / 73,100 rows
Inserted 60,000 / 73,100 rows
Inserted 65,000 / 73,100 rows
Inserted 70,000 / 73,100 rows
Inserted 73,100 / 73,100 rows

Data loading completed successfully!


In [82]:
cursor.execute("""
    SELECT COUNT(*)
    FROM inventory_sales
""")

count = cursor.fetchone()[0]

print(f"Rows in MySQL: {count:,}")

Rows in MySQL: 73,100
